# DPE neural combiners on frozen prediction bundles

Project adaptation inspired by **Regularized Neural Ensemblers (AutoML 2025)**, not a reproduction of that paper.

Compare uniform, logistic stacking, an 8-unit MLP, and an 8-unit softmax gate; MLP and gate each have a prediction-dropout variant. Reuse the same backbones, heads and A/B/C partitions from the completed H2 study. **CPU is sufficient.** Attach the previous combined study ZIP containing `source_archives/`, individual original H2 ZIPs, or extracted runs.

The default freezes choices using B/C only. Set `EVALUATE_TEST=True` to score the frozen family/overall winners. All planned seeds are frozen before any test cache is read. Results remain exploratory because this test set has already informed follow-up design. Keep the source bundles separately; the new download contains models and reports, not duplicate source predictions.


In [ ]:
SOURCE = None       # Previous study ZIP, original H2 ZIP/directory, or list of paths.
SEEDS = [0, 1, 2]    # Explicitly set [0] for a one-seed pilot.
TRACK = "standard"
EVALUATE_TEST = False
CONFIG_PATH = "configs/dpe_head_reweighting/neural_combiners.yaml"
REPO_PATH = None
WORK_DIR = None
SEARCH_PATHS = None
REPO_URL = "https://github.com/chautanphat/waterbirds-shortcut-mitigation.git"
REPO_REF = "experiment/dpe-neural-combiner"  # Publish this branch before using a fresh remote clone.


In [ ]:
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys

working = Path.cwd().resolve()
PLATFORM = "kaggle" if Path("/kaggle/working").is_dir() else "colab" if Path("/content").is_dir() else "local"
def compatible(path):
    return (path / "src/methods/dpe_head_reweighting/neural_study.py").is_file()
candidates = [working, *working.parents, working / "waterbirds-shortcut-mitigation"]
REPO = Path(REPO_PATH).expanduser().resolve() if REPO_PATH else next((p for p in candidates if compatible(p)), None)
if WORK_DIR is None:
    base = Path("/kaggle/working") if PLATFORM == "kaggle" else Path("/content") if PLATFORM == "colab" else (REPO / "outputs" if REPO else working)
    WORK_DIR = base / "dpe_neural_combiners_session"
WORK_DIR = Path(WORK_DIR).expanduser().resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
if REPO is None:
    REPO = WORK_DIR / "repository-neural-v1"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True)
if not compatible(REPO):
    raise FileNotFoundError("Use a checkout containing neural_study.py; set REPO_PATH to the implementation branch.")
missing = [package for module, package in [("numpy", "numpy>=1.26,<3"), ("scipy", "scipy>=1.11,<2"),
    ("torch", "torch>=2.6,<3"), ("yaml", "PyYAML>=6,<7")] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import display, FileLink
from src.methods.dpe_head_reweighting.neural_study import run_study
print("Platform:", PLATFORM, "| Repository:", REPO, "| Workspace:", WORK_DIR)


## Fit on B and freeze on C

The fixed YAML declares 400 full-batch CPU float64 AdamW steps, group-balanced BCE, weight decay 0.01/0.001 and prediction dropout 0/0.25. There are 11 candidates. Uniform is the fallback in every family. C selects WGA, then overall accuracy; complete ties retain uniform, then declaration order. There is no early stopping or B+C refit.

The runner preserves input bundles, accepts nested source archives in the previous combined H2 study ZIP, and fails if any requested seed is missing. It does not retrain backbones/heads. Re-running verifies saved model states and choices. Changed code/configuration/input identities produce a separate output directory.


In [ ]:
if SEARCH_PATHS is None:
    SEARCH_PATHS = [REPO / "outputs/dpe_head_reweighting_runs"]
    if PLATFORM == "kaggle":
        SEARCH_PATHS += [Path("/kaggle/input")]
    elif PLATFORM == "colab":
        SEARCH_PATHS += list(Path("/content").glob("*.zip")) + [Path("/content/uploads")]
    else:
        SEARCH_PATHS += list(working.glob("*.zip"))
STUDY = run_study(WORK_DIR, source=SOURCE, search_roots=SEARCH_PATHS, track=TRACK,
                  seeds=SEEDS, config_path=CONFIG_PATH, evaluate_test=EVALUATE_TEST)
output = Path(STUDY["output_dir"])
for seed in SEEDS:
    selection = json.loads((output / "comparisons" / f"seed{seed}" / "selection.json").read_text())
    print("Seed", seed, "overall C choice:", selection["selected"])
    print("Family choices:", selection["per_family_selected"])
    print("B/C complementarity (label-aware oracle is not a predictor):", selection["complementarity"])
    print((output / "comparisons" / f"seed{seed}" / "candidate_comparison.csv").read_text())
if STUDY["summary"] is None:
    print("Selections saved. Test evaluation was not requested. Set EVALUATE_TEST=True and rerun to evaluate.")
else:
    print(json.dumps(STUDY["summary"], indent=2))


## Download models and reports

The ZIP contains the fixed plan, exact JSON model states, all C candidates and, when requested, only frozen-winner test comparisons. Test outputs include per-group accuracy/counts, paired deltas, changed error IDs and gate concentration. Retain the original H2 prediction bundles separately for replay. No synthetic check is a Waterbirds result.


In [ ]:
archive = Path(STUDY["archive"])
previous = Path.cwd()
try:
    os.chdir(working)
    display(FileLink(os.path.relpath(archive, working), result_html_prefix="Download neural comparison: "))
finally:
    os.chdir(previous)
if PLATFORM == "colab":
    from google.colab import files
    files.download(str(archive))
